# NB-05: Figure & Image Dependency Checker

Scans all `\\includegraphics` calls in the HypatiaX paper bundle, detects `\\fbox`
placeholders, checks whether required image files exist on disk, and audits every
figure environment for labels and captions.

**Source files covered:**
- `jmlr_paper_main.tex` — main paper (5 figures)
- `supp_routing_improvements.tex` — Supplementary A (0 `\\includegraphics`)
- `supp_benchmark_report.tex` — Supplementary B (0 `\\includegraphics`; 13 figures listed in inventory)

**Steps:**
1. Extract all `\\includegraphics` paths from all three files
2. Detect `\\fbox` placeholder figures
3. Check image files — primary search in `hypatia/data/results/figures/`, then inventory paths
4. Full figure environment audit
5. Figures **not produced by runners** (hand-crafted / architecture diagrams)
6. Copy figures-inventory files → `$ROOT/figures/`
7. Fix recipe summary

## Step 0 — Configuration

In [ ]:
import re
import shutil
from pathlib import Path

# ── Source files ───────────────────────────────────────────────────────────────
TEX_FILES = [
    "jmlr_paper_main.tex",
    "supp_routing_improvements.tex",
    "supp_benchmark_report.tex",
]

# Root of the repository / project tree
# Adjust if running from a subdirectory, e.g. ROOT = Path("..") or Path("/path/to/repo")
ROOT = Path(".")

# ── Primary search: experiment runner output ───────────────────────────────────
# Repository layout from supp_routing_improvements.tex §Reproducibility:
#   papers/2025-JMLR/hypatiax/data/results/
RUNNER_FIGURES_DIR = ROOT / "hypatiax" / "data" / "results" / "figures"

# ── Figures inventory: hand-crafted / cosmetic files NOT produced by runners ───
# Sources inferred from NB-05 Step 5 fix recipe + supp_benchmark_report figure list
# All non-runner figures live in a single flat ROOT/Figures/ directory
# (no per-group subfolders). Generation paths:
#   - hypatiaX_three_systems, hypatiaX_algorithm1_routing_cascade_v2:
#       hand-crafted, no generator — must be placed in Figures/ manually.
#   - fig09_r2_heatmap_regimes, fig18_r2_heatmap_improved, fig1_seed_sweep:
#       generated by `python scripts/generate_figures.py --experiment exp1
#       --results-dir <exp1 results dir> --figures-dir <exp1 figures dir>
#       --source auto`, then synced to Figures/ by ci_postprocess figures_deploy.
#   - the 13 Supp-B sweep figures below: generated by
#       `python scripts/generate_figures.py --experiment suppB|suppB_sc ...`,
#       then synced to Figures/ by ci_postprocess figures_deploy.
FIGURES_INVENTORY = {
    # ── Main paper (jmlr_paper_main.tex) ──────────────────────────
    # fig:architecture §7.1 — currently \fbox placeholder
    "hypatiaX_three_systems": ROOT / "Figures" / "hypatiaX_three_systems.pdf",
    # fig:routing_cascade §7.4 — algorithm flow diagram
    "hypatiaX_algorithm1_routing_cascade_v2": ROOT / "Figures" / "hypatiaX_algorithm1_routing_cascade_v2.pdf",
    # fig:r2_heatmap_clipped §10.2 — cosmetic heatmap
    "fig18_r2_heatmap_improved": ROOT / "Figures" / "fig18_r2_heatmap_improved.pdf",
    # fig:r2_heatmap_raw §10.2 — raw heatmap
    "fig09_r2_heatmap_regimes": ROOT / "Figures" / "fig09_r2_heatmap_regimes.pdf",
    # fig:portfolio_seed_sweep §10.5 — seed sweep bar chart
    "fig1_seed_sweep": ROOT / "Figures" / "fig1_seed_sweep.pdf",

    # ── Supplementary B (supp_benchmark_report.tex) — full figure inventory ─
    # All 13 figures listed in Table A.1 of supp_benchmark_report.tex.
    # Generated by scripts/generate_figures.py --experiment suppB / suppB_sc
    # from the noise-sweep and sample-complexity result JSONs.
    "fig1_r2_vs_noise":        ROOT / "Figures" / "fig1_r2_vs_noise.pdf",
    "fig2_rmse_vs_noise":      ROOT / "Figures" / "fig2_rmse_vs_noise.pdf",
    "fig3_time_vs_noise":      ROOT / "Figures" / "fig3_time_vs_noise.pdf",
    "fig4_r2_vs_n":            ROOT / "Figures" / "fig4_r2_vs_n.pdf",
    "fig5_rmse_vs_n":          ROOT / "Figures" / "fig5_rmse_vs_n.pdf",
    "fig6_time_vs_n":          ROOT / "Figures" / "fig6_time_vs_n.pdf",
    "fig7_recovery_vs_noise":  ROOT / "Figures" / "fig7_recovery_vs_noise.pdf",
    "fig8_recovery_vs_n":      ROOT / "Figures" / "fig8_recovery_vs_n.pdf",
    "fig9_minr2_vs_noise":     ROOT / "Figures" / "fig9_minr2_vs_noise.pdf",
    "fig10_r2_boxplot_noise":  ROOT / "Figures" / "fig10_r2_boxplot_noise.pdf",
    "fig11_recovery_heatmap":  ROOT / "Figures" / "fig11_recovery_heatmap.pdf",
    "fig_runtime_comparison":  ROOT / "Figures" / "fig_runtime_comparison.png",
    "fig_comparative_table":   ROOT / "Figures" / "fig_comparative_table.png",
}

# ── Destination that LaTeX sees via \graphicspath{{figures/}{../figures/}} ─────
DEST_DIR = ROOT / "figures"

# ── Fallback search directories ────────────────────────────────────────────────
FALLBACK_DIRS = [
    ROOT / "figures",
    ROOT / "Figures",
    ROOT,
]

EXTENSIONS = [".pdf", ".png", ".jpg", ".eps", ".svg"]

# ── Load all source files ──────────────────────────────────────────────────────
sources = {}
for tex in TEX_FILES:
    p = Path(tex)
    if p.exists():
        sources[tex] = p.read_text(encoding="utf-8")
        print(f"  Loaded  {tex}  ({len(sources[tex].splitlines())} lines)")
    else:
        print(f"  MISSING {tex}  — not found on disk")

combined_source = "\n".join(sources.values())

print()
print(f"Runner figures dir  : {RUNNER_FIGURES_DIR.resolve()}")
print(f"Destination dir     : {DEST_DIR.resolve()}")
print(f"Inventory entries   : {len(FIGURES_INVENTORY)}")

## Step 1 — Extract all \\includegraphics paths

In [ ]:
INCL_RE = re.compile(r'\\includegraphics(?:\[[^\]]*\])?\{([^}]+)\}')

# Per-file breakdown
per_file_refs = {}
for tex, src in sources.items():
    refs = INCL_RE.findall(src)
    per_file_refs[tex] = refs

all_refs = [r for refs in per_file_refs.values() for r in refs]
unique_refs = sorted(set(all_refs))

print(f"Total \\includegraphics calls across all files: {len(all_refs)}")
print(f"Unique stems: {len(unique_refs)}")
print()

for tex, refs in per_file_refs.items():
    print(f"  {tex}  ({len(refs)} call{'s' if len(refs)!=1 else ''})")
    for r in refs:
        print(f"    {r}")
    if not refs:
        print(f"    (none)")
    print()

# Note: supp_benchmark_report.tex lists 13 figures in its Table A.1 figure inventory
# but does NOT use \includegraphics (figures are referenced only in the inventory table).
# Those stems are still tracked in FIGURES_INVENTORY and checked in Steps 3/6.
print("NOTE: supp_benchmark_report.tex lists 13 sweep figures in its Table A.1")
print("      inventory but does not embed them via \\includegraphics.")
print("      They are tracked in FIGURES_INVENTORY and are (re)generated by")
print("      scripts/generate_figures.py --experiment suppB / suppB_sc")
print("      (run via ci_postprocess.yml figures_deploy).")

## Step 2 — Detect \\fbox placeholder figures

In [ ]:
print("\\fbox occurrences across all source files:")
print("=" * 70)

total_fbox = 0
for tex, src in sources.items():
    lines = src.splitlines()
    fbox_lines = [(i+1, ln.strip()) for i, ln in enumerate(lines) if r'\fbox' in ln]
    total_fbox += len(fbox_lines)
    if fbox_lines:
        print(f"\n  {tex}  — {len(fbox_lines)} occurrence(s):")
        for lno, ctx in fbox_lines:
            print(f"    line {lno:4d}: {ctx[:110]}")

print(f"\nTotal \\fbox occurrences: {total_fbox}")
print()
print("ANALYSIS:")
print("  jmlr_paper_main.tex line 113 — \\fbox is the runtime-correction")
print("  notice box (text, not a figure placeholder). The architecture figure")
print("  (hypatiaX_three_systems) is already using \\includegraphics but the image")
print("  file is MISSING on disk — see Step 3.")

## Step 3 — Check required image files

Search priority:
1. `hypatia/data/results/figures/` — runner output
2. `FIGURES_INVENTORY` paths — hand-crafted / cosmetic figures
3. Fallback directories (`figures/`, `Figures/`, root)

All five main-paper stems **plus** the 13 Supplementary-B sweep figures are checked.
A **MISSING FIGURES SUMMARY** is printed at the end.

In [ ]:
def find_image(stem: str):
    """Return (found, path, source_label) for a figure stem."""
    # 1. Runner output
    for ext in EXTENSIONS:
        p = RUNNER_FIGURES_DIR / (stem + ext)
        if p.exists():
            return True, p, "runner"
    # 2. Inventory path
    inv_path = FIGURES_INVENTORY.get(stem)
    if inv_path and inv_path.exists():
        return True, inv_path, "inventory"
    # 3. Fallback directories
    for d in FALLBACK_DIRS:
        for ext in EXTENSIONS:
            p = d / (stem + ext)
            if p.exists():
                return True, p, "fallback"
    return False, None, "—"


# Build full required-image registry:
#   (a) stems actually embedded via \includegraphics in any tex file
#   (b) stems listed only in FIGURES_INVENTORY (e.g. Supplementary B figures)
EMBEDDED_STEMS = set(all_refs)          # from Step 1
INVENTORY_STEMS = set(FIGURES_INVENTORY.keys())
ALL_STEMS = EMBEDDED_STEMS | INVENTORY_STEMS

STEM_DESCRIPTIONS = {
    # Main paper
    "hypatiaX_three_systems":               "fig:architecture §7.1 — architecture diagram  [EMBEDDED, \\fbox placeholder]",
    "hypatiaX_algorithm1_routing_cascade_v2": "fig:routing_cascade §7.4 — algorithm flow    [EMBEDDED]",
    "fig18_r2_heatmap_improved":             "fig:r2_heatmap_clipped §10.2 — clipped heatmap [EMBEDDED]",
    "fig09_r2_heatmap_regimes":              "fig:r2_heatmap_raw §10.2 — raw heatmap         [EMBEDDED]",
    "fig1_seed_sweep":                       "fig:portfolio_seed_sweep §10.5 — seed sweep      [EMBEDDED]",
    # Supplementary B sweep figures (inventory only, not \includegraphics)
    "fig1_r2_vs_noise":        "Supp-B Fig1 — Median R² vs σ               [INVENTORY]",
    "fig2_rmse_vs_noise":      "Supp-B Fig2 — Median RMSE vs σ             [INVENTORY]",
    "fig3_time_vs_noise":      "Supp-B Fig3 — Avg time vs σ                [INVENTORY]",
    "fig4_r2_vs_n":            "Supp-B Fig4 — Median R² vs n               [INVENTORY]",
    "fig5_rmse_vs_n":          "Supp-B Fig5 — Median RMSE vs n             [INVENTORY]",
    "fig6_time_vs_n":          "Supp-B Fig6 — Median time vs n             [INVENTORY]",
    "fig7_recovery_vs_noise":  "Supp-B Fig7 — Recovery rate vs σ           [INVENTORY]",
    "fig8_recovery_vs_n":      "Supp-B Fig8 — Recovery rate vs n           [INVENTORY]",
    "fig9_minr2_vs_noise":     "Supp-B Fig9 — Min R² vs σ                  [INVENTORY]",
    "fig10_r2_boxplot_noise":  "Supp-B Fig10 — Per-eq R² box plots         [INVENTORY]",
    "fig11_recovery_heatmap":  "Supp-B Fig11 — Recovery heatmap σ×n        [INVENTORY]",
    "fig_runtime_comparison":  "Supp-B — Runtime bar chart, 6 methods      [INVENTORY]",
    "fig_comparative_table":   "Supp-B — Domain×method comparison table    [INVENTORY]",
}

missing_figures = []
found_figures   = []

print("Image file availability check:")
print("=" * 95)
for stem in sorted(ALL_STEMS):
    desc = STEM_DESCRIPTIONS.get(stem, f"[unknown stem — not in descriptions]")
    found, path, src = find_image(stem)
    if found:
        print(f"  [OK]       {stem}")
        print(f"             {desc}")
        print(f"             Source : {src}  →  {path}")
        found_figures.append(stem)
    else:
        inv_path = FIGURES_INVENTORY.get(stem)
        print(f"  [MISSING]  {stem}")
        print(f"             {desc}")
        print(f"             Runner : {RUNNER_FIGURES_DIR / (stem + '.pdf')}")
        print(f"             Inventory: {inv_path if inv_path else '(not in inventory)'}")
        missing_figures.append((stem, desc))
    print()

print("=" * 95)
print(f"MISSING FIGURES SUMMARY  ({len(missing_figures)} of {len(ALL_STEMS)} total):")
print()
if missing_figures:
    embedded_missing = [(s,d) for s,d in missing_figures if s in EMBEDDED_STEMS]
    inventory_missing = [(s,d) for s,d in missing_figures if s not in EMBEDDED_STEMS]
    if embedded_missing:
        print(f"  ✗ EMBEDDED figures missing from disk ({len(embedded_missing)}) — will break LaTeX build:")
        for stem, desc in embedded_missing:
            print(f"      {stem}")
            print(f"        {desc}")
        print()
    if inventory_missing:
        print(f"  ✗ Inventory-only figures missing ({len(inventory_missing)}) — needed for Supplementary B:")
        for stem, desc in inventory_missing:
            print(f"      {stem}")
else:
    print("  ✓ All figures accounted for.")

## Step 4 — Full figure environment audit

Audits every `\\begin{figure}` … `\\end{figure}` block across all three tex files.

In [ ]:
FIG_ENV_RE = re.compile(
    r'\\begin\{figure\*?\}(.*?)\\end\{figure\*?\}',
    re.DOTALL
)

total_figs = 0
for tex, src in sources.items():
    fig_blocks = FIG_ENV_RE.findall(src)
    if not fig_blocks:
        continue
    print(f"\n{'='*80}")
    print(f"  {tex}  —  {len(fig_blocks)} figure environment(s)")
    print(f"{'='*80}")
    for i, block in enumerate(fig_blocks):
        label_m = re.search(r'\\label\{([^}]+)\}', block)
        cap_m   = re.search(r'\\caption\{([^}]{0,80})', block)
        incl_m  = re.search(r'\\includegraphics(?:\[[^\]]*\])?\{([^}]+)\}', block)
        fbox_m  = r'\fbox' in block

        label = label_m.group(1) if label_m else "[NO LABEL]"
        cap   = cap_m.group(1).strip()[:72] if cap_m else "[NO CAPTION]"
        img   = incl_m.group(1) if incl_m else ("[fbox PLACEHOLDER]" if fbox_m else "[NO IMAGE]")

        # Determine status
        if not incl_m and not fbox_m:
            flag = "WARNING — NO IMAGE"
        elif not incl_m and fbox_m:
            flag = "WARNING — fbox PLACEHOLDER"
        else:
            found, _, _ = find_image(img)
            flag = "OK" if found else "WARNING — FILE MISSING"

        print(f"  Fig {total_figs+i+1:2d}  [{flag}]")
        print(f"          label  : {label}")
        print(f"          image  : {img}")
        print(f"          caption: {cap}…")
        print()
    total_figs += len(fig_blocks)

print(f"\nTotal figure environments across all files: {total_figs}")

## Step 5 — Figures NOT produced by runners

Cross-references each required stem against `RUNNER_FIGURES_DIR`.
Anything absent from the runner output must be placed manually
(architecture diagrams, cosmetic edits, sweep plots from JSON).

Figures not produced by an experiment runner fall into two groups:
- Hand-crafted, no generator: `hypatiaX_three_systems`, `hypatiaX_algorithm1_routing_cascade_v2`
  — must be placed in `Figures/` manually.
- Script-generated: `fig09_r2_heatmap_regimes`, `fig18_r2_heatmap_improved`, `fig1_seed_sweep`
  (from `--experiment exp1`) and the 13 Supp-B sweep figures
  (from `--experiment suppB` / `suppB_sc`) are regenerated via:
```bash
python scripts/generate_figures.py --experiment exp1     --results-dir <exp1 results>     --figures-dir <exp1 results>/figures     --source auto
python scripts/generate_figures.py --experiment suppB    --results-dir <suppB results>    --figures-dir <suppB results>/figures    --source auto
python scripts/generate_figures.py --experiment suppB_sc --results-dir <suppB_sc results> --figures-dir <suppB_sc results>/figures --source auto
```
(this is exactly what `ci_postprocess.yml`'s `figures_deploy` job does before
copying everything into `Figures/` and then `figures/`).

In [ ]:
print("Figures NOT produced by experiment runners")
print("(must be placed manually or regenerated from JSON)")
print("=" * 80)

non_runner = []
runner_produced = []

for stem in sorted(ALL_STEMS):
    runner_has = any(
        (RUNNER_FIGURES_DIR / (stem + ext)).exists()
        for ext in EXTENSIONS
    )
    if runner_has:
        runner_produced.append(stem)
    else:
        inv_path = FIGURES_INVENTORY.get(stem)
        if inv_path is not None and inv_path.exists():
            status = f"inventory → {inv_path}"
        elif inv_path is not None:
            status = f"MISSING — inventory path does not exist: {inv_path}"
        else:
            status = "MISSING — not in runner OR inventory"
        non_runner.append((stem, status))

# Categorise non-runner figures
arch_figs   = [s for s,_ in non_runner if "architecture" in s or "routing_cascade" in s or "three_systems" in s]
heatmap_figs= [s for s,_ in non_runner if "heatmap" in s or "r2_heatmap" in s]
sweep_figs  = [s for s,_ in non_runner if s.startswith("fig") and s not in arch_figs + heatmap_figs]

categories = [
    ("Architecture / algorithm diagrams (hand-crafted, no generator)", arch_figs),
    ("Cosmetic heatmaps (generated by scripts/generate_figures.py --experiment exp1)", heatmap_figs),
    ("Sweep & comparative plots (generated by scripts/generate_figures.py --experiment suppB / suppB_sc)", sweep_figs),
]

for cat_name, stems in categories:
    matching = [(s, st) for s, st in non_runner if s in stems]
    if not matching:
        continue
    print(f"\n  [{cat_name}]")
    for stem, status in matching:
        print(f"    {stem}")
        print(f"      → {status}")

print()
print(f"Non-runner total : {len(non_runner)} of {len(ALL_STEMS)}")
if runner_produced:
    print(f"Runner-produced  : {len(runner_produced)} — {runner_produced}")

# Regeneration commands
print()
print("Regeneration commands (run from repo root, or via ci_postprocess figures_deploy):")
print("  Cosmetic (fig09, fig18, fig1_seed_sweep):")
print("    python scripts/generate_figures.py --experiment exp1 --results-dir <exp1 results> \\")
print("      --figures-dir <exp1 results>/figures --source auto")
print("  Supplementary-B sweep figures (13 stems):")
print("    python scripts/generate_figures.py --experiment suppB --results-dir <suppB results> \\")
print("      --figures-dir <suppB results>/figures --source auto")
print("    python scripts/generate_figures.py --experiment suppB_sc --results-dir <suppB_sc results> \\")
print("      --figures-dir <suppB_sc results>/figures --source auto")

## Step 5a-debug — Environment & path diagnostics

Run **before** the Step 5b recovery guard. Prints the kernel's actual cwd,
where `ROOT` resolves to, and the real contents of every directory the
recovery/search logic depends on — at the notebook's own level *and* one
level up — so a CI failure shows exactly which directory is empty/missing
and why, instead of just the generic "both empty or missing" message.

Purely diagnostic: never raises, never modifies anything on disk.

In [ ]:
# ── Step 5a-debug: environment & path diagnostics (read-only, never raises) ──
import os
import sys
from pathlib import Path

def _ls(p: Path, max_items: int = 25):
    """Best-effort directory listing; never throws."""
    try:
        if not p.exists():
            return "<does not exist>"
        if not p.is_dir():
            return "<exists but is not a directory>"
        items = sorted(os.listdir(p))
        if not items:
            return "<exists, empty>"
        shown = items[:max_items]
        suffix = f"  (+{len(items) - max_items} more)" if len(items) > max_items else ""
        return ", ".join(shown) + suffix
    except Exception as e:
        return f"<error listing: {e!r}>"

def _tree(start: Path, depth: int = 1, prefix: str = ""):
    """Best-effort shallow directory tree; never throws."""
    if depth < 0:
        return
    try:
        entries = sorted(start.iterdir(), key=lambda p: (not p.is_dir(), p.name))
    except Exception as e:
        print(f"{prefix}<error: {e!r}>")
        return
    for entry in entries:
        marker = "/" if entry.is_dir() else ""
        print(f"{prefix}{entry.name}{marker}")
        if entry.is_dir() and depth > 0:
            _tree(entry, depth - 1, prefix + "    ")

def print_path_diagnostics(header="STEP 5a-debug — environment & path diagnostics"):
    """Print cwd, ROOT resolution, and every candidate dir Step 5b depends on,
    both at ROOT and one level up (PARENT). Read-only; never raises — safe to
    call proactively (Step 5a-debug) AND reactively right before a RuntimeError
    in Step 5b, so the diagnostics are guaranteed to land in the CI log even
    if nbconvert never writes the executed notebook back to disk.
    """
    print("=" * 80)
    print(header)
    print("=" * 80)

    print(f"\n[cwd]")
    print(f"  os.getcwd()        = {os.getcwd()}")
    print(f"  Path('.').resolve()= {Path('.').resolve()}")
    print(f"  sys.argv[0]        = {sys.argv[0] if sys.argv else '<empty>'}")

    print(f"\n[ROOT as configured in Step 0]")
    print(f"  ROOT (raw)         = {ROOT!r}")
    print(f"  ROOT.resolve()     = {ROOT.resolve()}")

    candidates = {
        "ROOT/figures":                        ROOT / "figures",
        "ROOT/tables":                         ROOT / "tables",
        "ROOT/Figures":                        ROOT / "Figures",
        "ROOT/hypatiax/data/results":          ROOT / "hypatiax" / "data" / "results",
        "ROOT/hypatiax/data/results/figures":  ROOT / "hypatiax" / "data" / "results" / "figures",
    }

    # One level UP from ROOT — catches the classic "nbconvert's kernel cwd is
    # the notebook's own directory, but figures/tables were deposited one
    # level up at the repo root" mismatch.
    PARENT = ROOT.resolve().parent
    candidates_parent = {
        "PARENT/figures":                       PARENT / "figures",
        "PARENT/tables":                        PARENT / "tables",
        "PARENT/Figures":                       PARENT / "Figures",
        "PARENT/hypatiax/data/results":         PARENT / "hypatiax" / "data" / "results",
        "PARENT/hypatiax/data/results/figures": PARENT / "hypatiax" / "data" / "results" / "figures",
        "PARENT/notebooks":                     PARENT / "notebooks",
    }

    print(f"\n[Candidates relative to ROOT = {ROOT.resolve()}]")
    for label, path in candidates.items():
        exists = path.exists()
        kind = "dir" if path.is_dir() else ("file" if path.is_file() else "-")
        print(f"  {label:38s} exists={exists!s:5s} kind={kind:4s} -> {_ls(path)}")

    print(f"\n[Same candidates one level up, PARENT = {PARENT}]")
    print( "  (catches: nbconvert's kernel cwd == notebook's own dir, while CI")
    print( "   populated figures/ or tables/ one level up at the repo root)")
    for label, path in candidates_parent.items():
        exists = path.exists()
        kind = "dir" if path.is_dir() else ("file" if path.is_file() else "-")
        print(f"  {label:38s} exists={exists!s:5s} kind={kind:4s} -> {_ls(path)}")

    print(f"\n[Directory tree around cwd, depth 1]")
    _tree(Path(".").resolve(), depth=1)

    print("\n" + "=" * 80)
    print("END " + header)
    print("=" * 80)

# Run once now, proactively, before Step 5b's recovery guard.
print_path_diagnostics()

## Step 5b — Pre-audit recovery: verify `figures/` and `tables/` are populated

Before the audit (Steps 6–7), checks that `repo_root/figures/` and
`repo_root/tables/` each exist and are non-empty.

If either directory is absent or empty, calls `repopulate_from_results()` to
gather figures/tables from **all** known source locations and copy them into
the repo-root flat directories — the same recovery logic that
`ci_paper_audit.yml` Job 0b performs:

- `ROOT/Figures/` (capital F) — hand-crafted figures with no generator, plus
  any runner-generated figures already synced here by `ci_postprocess.yml`
  (`figures_deploy`). This is the primary source per `FALLBACK_DIRS` /
  `FIGURES_INVENTORY` in the config cell.
- `hypatiax/data/results/` — raw runner output (`.pdf`/`.png` figures,
  `.tex` tables), scanned recursively as a secondary source for cases where
  `figures_deploy` hasn't run yet.

If a directory is still empty after recovery, a `RuntimeError` is raised so the
notebook stops immediately rather than producing a misleading audit.
Steps 6–7 never audit `hypatiax/data/results/` directly.

In [ ]:
# ── Step 5b: Pre-audit recovery — ensure repo_root/figures/ and repo_root/tables/ are populated ──
import shutil

FIG_DIR = ROOT / "figures"
TAB_DIR = ROOT / "tables"

def repopulate_from_results():
    """Copy figures (pdf/png) into repo_root/figures/ and tables (.tex) into
    repo_root/tables/, pulling from every known source location:
      1. ROOT/Figures/        - hand-crafted figures (no generator) + anything
                                 already synced here by ci_postprocess.yml
                                 figures_deploy. Primary source.
      2. hypatiax/data/results/ - raw runner output, scanned recursively.
                                 Secondary source for results not yet synced.
    Mirrors what ci_postprocess.yml 'Copy figures & tables' and
    ci_paper_audit Job 0b perform.
    """
    FIGURES_SRC = ROOT / "Figures"
    OUT_BASE = ROOT / "hypatiax" / "data" / "results"
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    TAB_DIR.mkdir(parents=True, exist_ok=True)

    fig_count = tab_count = 0

    # ── Source 1: ROOT/Figures/ (hand-crafted + already-synced figures) ──
    if FIGURES_SRC.exists():
        for p in sorted(FIGURES_SRC.rglob("*")):
            if not p.is_file():
                continue
            if p.suffix.lower() in (".pdf", ".png"):
                dest = FIG_DIR / p.name
                if not dest.exists():
                    shutil.copy2(p, dest)
                    fig_count += 1

    # ── Source 2: hypatiax/data/results/ (raw runner output) ──
    if OUT_BASE.exists():
        for p in sorted(OUT_BASE.rglob("*")):
            if not p.is_file():
                continue
            if p.suffix.lower() in (".pdf", ".png"):
                dest = FIG_DIR / p.name
                if not dest.exists():
                    shutil.copy2(p, dest)
                    fig_count += 1
            elif p.suffix.lower() == ".tex":
                exp_dir = p.parent.parent.name
                dest = TAB_DIR / f"{exp_dir}__{p.stem}.tex"
                if not dest.exists():
                    shutil.copy2(p, dest)
                    tab_count += 1

    print(f"  repopulate_from_results: {fig_count} figure(s), {tab_count} table(s) copied")
    print(f"    sources checked: {FIGURES_SRC} (exists={FIGURES_SRC.exists()}), "
          f"{OUT_BASE} (exists={OUT_BASE.exists()})")

if not FIG_DIR.exists() or not any(FIG_DIR.iterdir()):
    repopulate_from_results()

if not TAB_DIR.exists() or not any(TAB_DIR.iterdir()):
    repopulate_from_results()

if not any(FIG_DIR.iterdir()):
    print("\n!!! FIG_DIR empty after recovery — dumping diagnostics before raising !!!\n")
    print_path_diagnostics(header="STEP 5b-FAILURE — FIG_DIR empty, diagnostics at point of failure")
    raise RuntimeError(
        f"{FIG_DIR} is empty after recovery — "
        f"checked {ROOT / 'Figures'} and {ROOT / 'hypatiax' / 'data' / 'results'}, both empty or missing. "
        "Place figures in Figures/ or run ci_postprocess.yml (figures_deploy), "
        "then re-run this notebook. See STEP 5b-FAILURE diagnostics above for the "
        "actual cwd/ROOT resolution and what was really on disk."
    )

if not any(TAB_DIR.iterdir()):
    print("\n!!! TAB_DIR empty after recovery — dumping diagnostics before raising !!!\n")
    print_path_diagnostics(header="STEP 5b-FAILURE — TAB_DIR empty, diagnostics at point of failure")
    raise RuntimeError(
        f"{TAB_DIR} is empty after recovery — "
        "run ci_postprocess.yml (any experiment) or commit .tex files to tables/ directly, "
        "then re-run this notebook. See STEP 5b-FAILURE diagnostics above for the "
        "actual cwd/ROOT resolution and what was really on disk."
    )

print(f"✅  figures/ : {sum(1 for _ in FIG_DIR.iterdir())} file(s)")
print(f"✅  tables/  : {sum(1 for _ in TAB_DIR.iterdir())} file(s)")
print("Pre-audit check PASSED — proceeding to Step 6.")


## Step 6 — Copy inventory figures → `$ROOT/figures/`

Copies every figure that exists at its `FIGURES_INVENTORY` path to `DEST_DIR`
(the `figures/` directory that LaTeX reads via `\graphicspath{{figures/}{../figures/}}`).

Set `DRY_RUN = True` to preview without writing anything. Default is `False` — copies are performed on run.

In [ ]:
DRY_RUN = False   # ← set True to preview without writing

DEST_DIR.mkdir(parents=True, exist_ok=True)

print(f"{'[DRY RUN] ' if DRY_RUN else ''}Copying inventory figures → {DEST_DIR.resolve()}")
print("=" * 80)

copied, skipped, not_found = [], [], []

for stem, src_path in FIGURES_INVENTORY.items():
    if not src_path.exists():
        print(f"  [NOT FOUND]  {stem}")
        print(f"               Expected : {src_path}")
        not_found.append(stem)
        continue

    dest_path = DEST_DIR / src_path.name
    if dest_path.exists():
        print(f"  [SKIP]       {src_path.name}  (already in {DEST_DIR})")
        skipped.append(stem)
        continue

    if not DRY_RUN:
        shutil.copy2(src_path, dest_path)
        print(f"  [COPIED]     {src_path}")
        print(f"               → {dest_path}")
    else:
        print(f"  [WOULD COPY] {src_path}")
        print(f"               → {dest_path}")
    copied.append(stem)
    print()

print("=" * 80)
print(f"Copied    : {len(copied)}")
print(f"Skipped   : {len(skipped)}  (already present)")
print(f"Not found : {len(not_found)}  (source missing — see Fix recipe below)")

if DRY_RUN and (copied or not_found):
    print()
    print("Set DRY_RUN = False and re-run this cell to perform the copies.")

## Step 7 — Fix recipe

Re-checks all stems after Step 6 (in case copies were performed) and prints
only the actions still required.

In [ ]:
# Fix details keyed by stem — derived from tex file content and NB-05 history
FIX_DETAILS = {
    "hypatiaX_three_systems": (
        "FIX-F1  hypatiaX_three_systems — Architecture figure (fig:architecture §7.1)\n"
        "  Status  : \\includegraphics present in main paper; image FILE MISSING on disk.\n"
        "  Source  : Figures/hypatiaX_three_systems.pdf  (hand-crafted, no generator)\n"
        "  ACTION  : Produce final PDF/PNG and place at the source path above,\n"
        "            then set DRY_RUN=False and re-run Step 6.\n"
    ),
    "hypatiaX_algorithm1_routing_cascade_v2": (
        "FIX-F2  hypatiaX_algorithm1_routing_cascade_v2 — Routing cascade (fig:routing_cascade §7.4)\n"
        "  Source  : Figures/hypatiaX_algorithm1_routing_cascade_v2.pdf  (hand-crafted, no generator)\n"
        "  NOTE    : Previously marked OK (B-07 addressed) — re-verify file compiles.\n"
        "  ACTION  : Confirm source exists; re-run Step 6.\n"
    ),
    "fig18_r2_heatmap_improved": (
        "FIX-F3  fig18_r2_heatmap_improved — R² heatmap clipped (fig:r2_heatmap_clipped §10.2)\n"
        "  Source  : Figures/fig18_r2_heatmap_improved.pdf\n"
        "  ACTION  : Run `python scripts/generate_figures.py --experiment exp1 ...`\n"
        "            (or ci_postprocess figures_deploy), then re-run Step 6.\n"
    ),
    "fig09_r2_heatmap_regimes": (
        "FIX-F4  fig09_r2_heatmap_regimes — R² heatmap raw (fig:r2_heatmap_raw §10.2)\n"
        "  Source  : Figures/fig09_r2_heatmap_regimes.pdf\n"
        "  ACTION  : Run `python scripts/generate_figures.py --experiment exp1 ...`\n"
        "            (or ci_postprocess figures_deploy), then re-run Step 6.\n"
    ),
    "fig1_seed_sweep": (
        "FIX-F5  fig1_seed_sweep — Portfolio Variance seed sweep (fig:portfolio_seed_sweep §10.5)\n"
        "  Source  : Figures/fig1_seed_sweep.pdf (or .png)\n"
        "  ACTION  : Run `python scripts/generate_figures.py --experiment exp1 ...`\n"
        "            (or ci_postprocess figures_deploy), then re-run Step 6.\n"
    ),
}
# Generic fix for Supplementary B sweep figures
SWEEP_STEMS = {
    "fig1_r2_vs_noise", "fig2_rmse_vs_noise", "fig3_time_vs_noise",
    "fig4_r2_vs_n", "fig5_rmse_vs_n", "fig6_time_vs_n",
    "fig7_recovery_vs_noise", "fig8_recovery_vs_n", "fig9_minr2_vs_noise",
    "fig10_r2_boxplot_noise", "fig11_recovery_heatmap",
    "fig_runtime_comparison", "fig_comparative_table",
}

print("FIX RECIPE — actions required before final LaTeX build")
print("=" * 80)

# Re-check after any copies
still_missing_embedded  = [(s, d) for s, d in missing_figures if s in EMBEDDED_STEMS and not find_image(s)[0]]
still_missing_inventory = [(s, d) for s, d in missing_figures if s not in EMBEDDED_STEMS and not find_image(s)[0]]

if not still_missing_embedded and not still_missing_inventory:
    print("  ✓ All figures resolved. No manual action required.")
else:
    if still_missing_embedded:
        print(f"\n── EMBEDDED figures (will BREAK the LaTeX build) ──────────────────────")
        for stem, desc in still_missing_embedded:
            msg = FIX_DETAILS.get(stem,
                f"FIX  {stem}\n  Description: {desc}\n"
                f"  ACTION: Locate file and copy to figures/.\n")
            print(msg)

    if still_missing_inventory:
        sweep_missing = [(s,d) for s,d in still_missing_inventory if s in SWEEP_STEMS]
        other_missing = [(s,d) for s,d in still_missing_inventory if s not in SWEEP_STEMS]

        if sweep_missing:
            print(f"\n── Supplementary B sweep figures (regenerate via generate_figures.py) ─")
            print(f"  Run:")
            print(f"    python scripts/generate_figures.py --experiment suppB \\")
            print(f"      --results-dir <suppB results> --figures-dir <suppB results>/figures --source auto")
            print(f"    python scripts/generate_figures.py --experiment suppB_sc \\")
            print(f"      --results-dir <suppB_sc results> --figures-dir <suppB_sc results>/figures --source auto")
            print(f"  (or run ci_postprocess.yml figures_deploy, which does both then copies to Figures/)")
            print(f"  Missing stems ({len(sweep_missing)}):")
            for stem, _ in sweep_missing:
                inv_path = FIGURES_INVENTORY.get(stem)
                print(f"    {stem}")
                print(f"      → expected at: {inv_path}")

        for stem, desc in other_missing:
            print(f"\nFIX  {stem}")
            print(f"  {desc}")
            print(f"  ACTION: Locate or regenerate; copy to figures/.")

print()
print("── Additional checks required ────────────────────────────────────────")
print("  1. Verify \\fbox at jmlr_paper_main.tex line ~113 is the")
print("     runtime-correction notice box (text), NOT a figure placeholder.")
print("     Confirmed: it is informational text, not a missing figure.")
print()
print("  2. supp_benchmark_report.tex references 13 figures in Table A.1 but")
print("     uses NO \\includegraphics — all 13 must be regenerated from v2 JSON.")
print()
print("  3. Citation key conflicts to resolve before JMLR submission")
print("     (flagged in supp_benchmark_report.tex Appendix):")
print("       • Unify citation key: udrescu2020aifeynman (both files)")
print("       • Replace meidani2023snip → meidani2024snip (ICLR 2024 key)")
print("       • Add missing @article{la2021contemporary, ...} to references.bib")